# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

# Optionally, inspect some additional general attributes
print(f"Identifier: {metadata.identifier}\nPublished: {metadata.datePublished}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we list all record set `@id`s and their fields as defined in the dataset. This helps us plan data extraction and analysis by always referencing entities by their `@id`.

In [ ]:
# List all record sets and their fields by `@id`
record_sets = list(metadata.record_sets)
print("Available Record Sets and Fields:")

for rs in record_sets:
    print(f'- Record Set: @id = {rs["@id"]}, name = {rs.get("name", "")}')
    if hasattr(rs, "fields"):
        for field in rs.fields:
            print(f'    - Field: @id = {field["@id"]}, name = {field.get("name", "")} (dataType: {field.get("dataType", "")})')
    elif isinstance(rs, dict) and "fields" in rs:
        # for robustness when 'fields' is a dict list, not attribute
        for field in rs["fields"]:
            print(f'    - Field: @id = {field["@id"]}, name = {field.get("name", "")} (dataType: {field.get("dataType", "")})')
    else:
        print('    (No fields described)')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

*Note: Please refer to the listing above for all available record sets. For this dataset, the main survey responses are typically stored in a record set with an `@id` ending in `/survey_recordset` or similar. Adjust the `record_set_ids` list as necessary for this dataset.*

In [ ]:
# Example: get all main record set ids (adjust as needed based on above listing)
record_set_ids = [rs["@id"] for rs in metadata.record_sets]
print("Will attempt to extract these record sets (by @id):", record_set_ids)

# Load all tables into dataframes, keyed by their `@id`
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from record set {rs_id}")

# Pick the main survey record set for demo
if len(record_set_ids) > 0:
    main_rs_id = record_set_ids[0]  # Use the first record set as default
    print(f"Main record set: {main_rs_id}")
    print("Fields:", list(dataframes[main_rs_id].columns))
    dataframes[main_rs_id].head()
else:
    print("No record sets found!")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below we:
- Select a numeric field (`phq9_score`, for example) by its field `@id`.
- Filter for high scores (e.g., PHQ-9 > 10).
- Normalize the numeric field.
- Group by another field (e.g., `level_of_education`).

Adapt the field `@id`s below to your dataset as needed.

In [ ]:
# Select a record set and fields by `@id` as listed in previous cell
selected_record_set = main_rs_id  # e.g., 'https://api.app.sen.science/frontiers/7160411/survey_main_records' (example only)
df = dataframes[selected_record_set]

# Choose a numeric field for analysis. Replace with appropriate field `@id` as needed.
possible_numeric_fields = [col for col in df.columns if "score" in col.lower() or df[col].dtype in [int, float]]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    numeric_field = df.columns[0]  # fallback

print(f"Using numeric field: {numeric_field}")

# Filter records where the value is above a threshold
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
print(filtered_df.head())

# Normalize
filtered_df[numeric_field + "_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

# Group by another field (e.g., 'level_of_education' or similar): pick the first string column that looks categorical
possible_group_fields = [col for col in df.columns if 'education' in col.lower() or 'gender' in col.lower() or 'marital' in col.lower()]
if possible_group_fields:
    group_field = possible_group_fields[0]
    print(f"Grouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
    print(grouped_df.head())
else:
    print("No obvious group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If we grouped above, plot
if 'group_field' in locals() and possible_group_fields:
    plt.figure(figsize=(10,5))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We demonstrated how to load and explore a Croissant dataset using the `mlcroissant` library.
- We identified multiple record sets and extracted tables using their `@id`. Fields were dynamically selected for demonstration.
- Basic filtering, normalization, grouping, and visualization steps were conducted on survey data, with all columns and keys referenced by their `@id`.

For further analysis, consult the field and record set listings in the overview section and adapt fields and filtering/grouping logic as needed for your research needs.